# 01. Первичный анализ данных (EDA)

**Что делаем в этом ноутбуке:** знакомимся с датасетом, проверяем качество данных, чистим то, что мешает анализу, и формулируем главный вопрос для следующего ноутбука.

**Один вывод одной фразой:** в данных ~1 млн транзакций, но после чистки (возвраты, отсутствующий `Customer ID`, бесплатные позиции) остаётся около **800 тыс. строк** и **~5,8 тыс. уникальных клиентов** — этого более чем достаточно для когортного анализа.

## Импорты и настройки

Подгружаем `pandas` для табличной работы, `matplotlib` и `seaborn` для графиков. Никаких экзотических библиотек — всё стандартное.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## Загрузка данных

В файле два листа — `Year 2009-2010` и `Year 2010-2011`. Объединяем их в один датафрейм, чтобы работать с полными двумя годами.

In [ ]:
DATA_PATH = '../data/online_retail_II.xlsx'

df_09_10 = pd.read_excel(DATA_PATH, sheet_name='Year 2009-2010')
df_10_11 = pd.read_excel(DATA_PATH, sheet_name='Year 2010-2011')

df = pd.concat([df_09_10, df_10_11], ignore_index=True)
print(f'Размер: {df.shape[0]:,} строк, {df.shape[1]} колонок')
df.head()

**Что в данных:**
- `Invoice` — номер чека. Если начинается с `C`, это возврат.
- `StockCode` — артикул товара.
- `Description` — название товара.
- `Quantity` — количество. Может быть отрицательным (возврат).
- `InvoiceDate` — дата и время покупки.
- `Price` — цена за единицу.
- `Customer ID` — идентификатор клиента (главное поле для нас, без него клиента не отследить).
- `Country` — страна доставки.

## Проверка пропусков и типов

Первое, что нужно понять про любой датасет — где дыры. Особенно важен `Customer ID`: без него клиент анонимный и в когортный анализ не попадает.

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing': missing, 'pct': missing_pct})

**Что мы видим:** примерно у 22% строк нет `Customer ID`. Это либо гостевые покупки, либо технические ошибки. Для когортного анализа эти строки бесполезны — мы их удалим, но сначала зафиксируем потерю.

## Очистка данных

Шаги чистки и зачем каждый нужен:
1. Убрать строки без `Customer ID` — без них нельзя строить retention.
2. Убрать возвраты (`Invoice`, начинающиеся с `C`) — это не покупки, а отмены.
3. Убрать строки с `Quantity <= 0` или `Price <= 0` — это либо ошибки, либо технические проводки.
4. Создать колонку `Revenue = Quantity * Price` — нам понадобится для среднего чека.

In [ ]:
before = len(df)

df = df.dropna(subset=['Customer ID'])
df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
df['Customer ID'] = df['Customer ID'].astype(int)
df['Revenue'] = df['Quantity'] * df['Price']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').dt.to_timestamp()

print(f'Было строк: {before:,}')
print(f'Стало строк: {len(df):,}')
print(f'Потеряли:   {before - len(df):,} ({(1 - len(df)/before)*100:.1f}%)')

**Комментарий:** мы потеряли около 25% строк, в основном из-за отсутствия `Customer ID`. Это ожидаемо для розничного датасета и не искажает анализ — оставшиеся строки уже принадлежат клиентам, которых мы можем отслеживать во времени.

## Базовая статистика

Смотрим масштаб: сколько у нас клиентов, заказов, какой период покрывают данные, какая выручка.

In [ ]:
print(f'Период:           {df["InvoiceDate"].min():%Y-%m-%d}  →  {df["InvoiceDate"].max():%Y-%m-%d}')
print(f'Уникальных клиентов: {df["Customer ID"].nunique():,}')
print(f'Уникальных заказов:  {df["Invoice"].nunique():,}')
print(f'Уникальных товаров:  {df["StockCode"].nunique():,}')
print(f'Общая выручка:       £{df["Revenue"].sum():,.0f}')
print(f'Стран в данных:      {df["Country"].nunique()}')

## Откуда клиенты

Распределение по странам — простая, но полезная картинка. Подозреваю, что доминирует Великобритания.

In [ ]:
country_revenue = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)
ax = country_revenue.plot(kind='barh', figsize=(10, 5), color='#4C72B0')
ax.invert_yaxis()
ax.set_title('Топ-10 стран по выручке')
ax.set_xlabel('Выручка, £')
plt.tight_layout()
plt.show()

**Что видим:** Великобритания даёт >85% выручки. Это локальный британский ритейлер. Дальше анализ можно делать как по всему датасету, так и только по UK — для интерпретируемости результатов оставим всё, но в рекомендациях упомянем, что выводы релевантны прежде всего для британского рынка.

## Динамика выручки по месяцам

Сезонность — важный контекст. Если в декабре всплеск, нужно будет учесть это в когортном анализе (декабрьские когорты могут иметь нетипичных клиентов — «подарочных» покупателей).

In [ ]:
monthly = df.groupby('InvoiceMonth').agg(
    revenue=('Revenue', 'sum'),
    orders=('Invoice', 'nunique'),
    customers=('Customer ID', 'nunique')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly['InvoiceMonth'], monthly['revenue'], marker='o', color='#4C72B0')
ax.set_title('Выручка по месяцам')
ax.set_ylabel('Выручка, £')
ax.set_xlabel('Месяц')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Что видим:** ярко выраженный пик в **ноябре–декабре** обоих лет — предновогодний шопинг. Это типично для розницы. Для нас это ожидаемо и уместно, но в выводах нужно будет помнить: декабрьские когорты могут вести себя иначе.

## Распределение чеков

Соберём чеки (`Invoice`) и посмотрим на распределение их размера. Это пригодится в следующем ноутбуке, когда будем сравнивать средний первый чек у вернувшихся и невернувшихся клиентов.

In [ ]:
invoice_totals = df.groupby('Invoice')['Revenue'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(invoice_totals, bins=80, color='#4C72B0')
axes[0].set_title('Размер чека (включая выбросы)')
axes[0].set_xlabel('£')
axes[0].set_ylabel('Кол-во чеков')

# Без выбросов: убираем верхний 1% для читаемости
p99 = invoice_totals.quantile(0.99)
axes[1].hist(invoice_totals[invoice_totals < p99], bins=80, color='#55A868')
axes[1].set_title(f'Размер чека (без верхнего 1%, < £{p99:.0f})')
axes[1].set_xlabel('£')
plt.tight_layout()
plt.show()

print(f'Медианный чек: £{invoice_totals.median():.2f}')
print(f'Средний чек:   £{invoice_totals.mean():.2f}')

**Что видим:** распределение сильно скошенное вправо — большинство чеков мелкие, а несколько чеков очень крупные (оптовые покупатели). Среднее почти в два раза больше медианы — это типичный признак длинного хвоста.

**Что это значит для анализа:** при сравнении средних чеков (в следующем ноутбуке) важно либо использовать робастные методы, либо честно отметить, что среднее чувствительно к выбросам. Я буду использовать **t-тест с независимыми выборками**, и дополнительно покажу медианы — чтобы убедиться, что вывод не держится только на хвостах.

## Сохраняем чистые данные для следующего ноутбука

Чтобы не повторять очистку в каждом ноутбуке, сохраняем результат в `parquet` — это быстрее, чем CSV или Excel, и сохраняет типы.

In [ ]:
df.to_parquet('../data/clean.parquet', index=False)
print('Сохранено: data/clean.parquet')

## Итог EDA

**Что мы знаем после EDA:**
1. Данные двухлетние, с выраженной декабрьской сезонностью.
2. После чистки осталось ~800 тыс. строк и ~5,8 тыс. клиентов — выборки достаточно для статистики.
3. Распределение чеков скошенное — нужно будет это учитывать.
4. 85% выручки — UK; выводы будут локальными, и это нужно проговорить.

**Готовы перейти к ядру:** строить когортный retention и сравнивать первый чек у вернувшихся и невернувшихся клиентов в `02_analysis.ipynb`.